# Isotropic plate with an offset

For a homogeneous isotropic plate of thickness $h$ whose mid-surface is at
$z = d$ from the reference surface, i.e. `offset = d`, the thickness integrals
have the closed form

$$
A = h C, \qquad B = d h C, \qquad D = h \left( d^2 + \frac{h^2}{12}
\right) C, \qquad A_{ts} = \frac{5}{6} G h I
$$

where $C$ is the plane-stress stiffness of the material and $I$ the 2x2
identity matrix. The offset creates a membrane-bending coupling $B \neq 0$
and, in the same way, the odd moments of the density, see
[Mass integrals](../mass_integrals.rst). The code below, the unit test
`test_example_isotropic_plate_offset` of `tests/test_core.py`, checks these
expressions, the matrices $E$, $F$, $H$ of the TSDT and the six mass integrals
against the integrals $\int_{d - h/2}^{d + h/2} z^n dz$.

## Where the closed form comes from

With the same $C$ in the whole thickness, the integrals of
[Laminate stiffness and plate theories](../constitutive_matrices.rst),

$$
(A, B, D, E, F, H) = C \int_{d - h/2}^{d + h/2} (1, z, z^2, z^3, z^4, z^6)
\, dz,
$$

are moments of $z$. Writing $z = d + s$, with $s$ uniform in $[-h/2, h/2]$,
$\int (d + s)^n ds = h \langle (d + s)^n \rangle$, where the odd moments of
$s$ vanish and $\langle s^2 \rangle = h^2/12$, $\langle s^4 \rangle = h^4/80$,
$\langle s^6 \rangle = h^6/448$. The first three give $A$, $B$ and $D$ above,
which is the shift of the reference surface of Vasiliev and Morozov [1], Eq.
(5.28), $B = B^0 + d A$ and $D = D^0 + 2 d B^0 + d^2 A$, with $B^0 = 0$ and
$D^0 = h^3 C/12$ about the mid-surface.

The transverse shear stiffness $\bar{A}_{ts} = G h I$ does not depend on the
reference surface. For a homogeneous plate the equilibrium distribution is the
parabola $\tau_{xz} = 3 Q_x/(2h) (1 - 4 \bar{z}^2/h^2)$, $\bar{z}$ measured from
the mid-surface, whose complementary energy $3 Q_x^2/(5 G h)$ equals the
energy $Q_x^2/(2 \kappa G h)$ of a constant distribution for $\kappa = 5/6$,
see [Shear correction](../shear_correction.rst). Rohwer [2], Sec. 4.1, p. 17,
states that this ratio "always amounts to 5/6, independent of the Poisson's
ratio or the position of the reference surface", which the test checks with
an offset.

All the integrals are computed exactly, ply by ply, so the tolerances are of
the order of round-off (`rtol=1e-12`).

In [1]:
import sys
sys.path.insert(0, '../tests') # helpers of the tests

import numpy as np

from composites.utils import isotropic_plate

In [2]:
# closed form of an isotropic plate whose mid-surface is at z = d
E, nu, h, d, rho = 70e9, 0.3, 2e-3, 0.6e-3, 2700.
lam = isotropic_plate(thickness=h, E=E, nu=nu, offset=d, rho=rho)
C = E/(1 - nu**2)*np.array([[1, nu, 0], [nu, 1, 0], [0, 0, (1 - nu)/2]])
G = E/(2*(1 + nu))
atol = 1e-12*E
assert np.allclose(lam.A, h*C, rtol=1e-12, atol=atol*h)
assert np.allclose(lam.B, d*h*C, rtol=1e-12, atol=atol*h*d)
assert np.allclose(lam.D, h*(d**2 + h**2/12)*C, rtol=1e-12, atol=atol*h**3)
# the shear correction of an isotropic plate is 5/6
assert np.allclose(lam.Ats, 5/6*G*h*np.eye(2), rtol=1e-12, atol=atol*h)
print('B =\n', lam.B)
print('d h C =\n', d*h*C)
print('Ats/(G h) =\n', lam.Ats/(G*h))

B =
 [[92307.69230769 27692.30769231     0.        ]
 [27692.30769231 92307.69230769     0.        ]
 [    0.             0.         32307.69230769]]
d h C =
 [[92307.69230769 27692.30769231     0.        ]
 [27692.30769231 92307.69230769     0.        ]
 [    0.             0.         32307.69230769]]
Ats/(G h) =
 [[ 0.83333333 -0.        ]
 [-0.          0.83333333]]


## Matrices of the TSDT and mass integrals

The higher moments follow from the same expansion,
$\langle (d + s)^3 \rangle = d^3 + d h^2/4$,
$\langle (d + s)^4 \rangle = d^4 + d^2 h^2/2 + h^4/80$ and
$\langle (d + s)^6 \rangle = d^6 + 15 d^4 h^2/12 + 15 d^2 h^4/80 + h^6/448$,
the list `m` below. They give $E = h \langle z^3 \rangle C$,
$F = h \langle z^4 \rangle C$, $H = h \langle z^6 \rangle C$, and the moments of
the density of [Mass integrals](../mass_integrals.rst),
$I_n = \int \rho z^n dz = \rho h \langle z^n \rangle$, stored as `intrho`,
`intrhoz`, `intrhoz2`, `intrhoz3`, `intrhoz4` and `intrhoz6`.

In [3]:
# with s uniform in [-h/2, h/2]: <s^2> = h^2/12, <s^4> = h^4/80,
# <s^6> = h^6/448, and int (d + s)^n ds = h <(d + s)^n>
m = [1., d, d**2 + h**2/12, d**3 + d*h**2/4,
     d**4 + d**2*h**2/2 + h**4/80, None,
     d**6 + 15/12*d**4*h**2 + 15/80*d**2*h**4 + h**6/448]
assert np.allclose(lam.E, h*m[3]*C, rtol=1e-12, atol=atol*h**4)
assert np.allclose(lam.F, h*m[4]*C, rtol=1e-12, atol=atol*h**5)
assert np.allclose(lam.H, h*m[6]*C, rtol=1e-12, atol=atol*h**7)
for name, n in (('intrho', 0), ('intrhoz', 1), ('intrhoz2', 2),
                ('intrhoz3', 3), ('intrhoz4', 4), ('intrhoz6', 6)):
    assert np.isclose(getattr(lam, name), rho*h*m[n], rtol=1e-12, atol=0)
    print('%-8s = %.6e, closed form %.6e' % (name, getattr(lam, name), rho*h*m[n]))

intrho   = 5.400000e+00, closed form 5.400000e+00
intrhoz  = 3.240000e-03, closed form 3.240000e-03
intrhoz2 = 3.744000e-06, closed form 3.744000e-06
intrhoz3 = 4.406400e-09, closed form 4.406400e-09
intrhoz4 = 5.667840e-12, closed form 5.667840e-12
intrhoz6 = 1.035457e-17, closed form 1.035457e-17


Without offset the density is symmetric about the reference surface, and the
odd moments $I_1$, $I_3$ and the coupling $B$ vanish:

In [4]:
# without offset the odd moments vanish
lam0 = isotropic_plate(thickness=h, E=E, nu=nu, rho=rho)
assert abs(lam0.intrhoz) < 1e-15*rho*h**2
assert abs(lam0.intrhoz3) < 1e-15*rho*h**4
assert np.allclose(lam0.B, 0, atol=atol*h**2)
print('intrhoz, intrhoz3 =', lam0.intrhoz, lam0.intrhoz3)

intrhoz, intrhoz3 = 0.0 0.0


## References

1. Vasiliev, V. V. and Morozov, E. V., Mechanics and Analysis of Composite Materials, Elsevier, Oxford, 2001.
2. Rohwer, K., Improved Transverse Shear Stiffnesses for Layered Finite Elements, DFVLR-FB 88-32, 1988.